In [9]:
import os
# Change working root directory
try:
    os.chdir("/detectives_minimal_implementation")
except: pass

In [10]:
# ! pip install booknlp_fr -U
from booknlp_fr import (
    load_tokenizer_and_embedding_model,
    get_embedding_tensor_from_tokens_df,
    load_text_file,
    load_tokens_df,
    save_tokens_df,
    load_entities_df,
)
from tqdm.auto import tqdm
import torch
import os
import pandas as pd

In [11]:
detective_annotated_dataset = pd.read_csv('data/detective_annotated_dataset.csv')
tokens_entities_files_directory = "data/texts"
embeddings_directory = "data/attribute_embeddings"

detectives_features_dataset = []

for file_name, file_df in tqdm(detective_annotated_dataset.groupby("text_id")):
    file_name = os.path.splitext(file_name)[0]

    attributes_embeddings_path = os.path.join(embeddings_directory, f"{file_name}.attribute_embeddings")
    if not os.path.exists(attributes_embeddings_path):
        print(f"No attributes embeddings found for {file_name}")
        continue

    attribute_embeddings = torch.load(attributes_embeddings_path)

    tokens_df = load_tokens_df(file_name, tokens_entities_files_directory)
    attributes_tokens_df = tokens_df[tokens_df["is_PER_attribute"] == 1].copy().reset_index(drop=True)

    entities_df = load_entities_df(file_name, tokens_entities_files_directory)

    for char_id, char_name, label in file_df[["char_id", "char_name", "label"]].values:
        character_entities_df = entities_df[entities_df["COREF"] == char_id].copy()
        character_head_ids = character_entities_df["head_id"].tolist()

        agent_attributes_ids = attributes_tokens_df[attributes_tokens_df["char_att_agent"].isin(character_head_ids)].index.tolist()
        patient_attributes_ids = attributes_tokens_df[attributes_tokens_df["char_att_patient"].isin(character_head_ids)].index.tolist()
        mod_attributes_ids = attributes_tokens_df[attributes_tokens_df["char_att_mod"].isin(character_head_ids)].index.tolist()
        pos_attributes_ids = attributes_tokens_df[attributes_tokens_df["char_att_poss"].isin(character_head_ids)].index.tolist()


        detectives_features_dataset.append(
            {"file_name": file_name,
             "char_name": char_name,
             "char_id": char_id,
             "label": label,
             "agent_lemmas": attributes_tokens_df.loc[agent_attributes_ids, "lemma"].tolist(),
             "patient_lemmas": attributes_tokens_df.loc[patient_attributes_ids, "lemma"].tolist(),
             "mod_lemmas": attributes_tokens_df.loc[mod_attributes_ids, "lemma"].tolist(),
             "pos_lemmas": attributes_tokens_df.loc[pos_attributes_ids, "lemma"].tolist(),
             "agent_embeddings": attribute_embeddings[agent_attributes_ids],
             "patient_embeddings": attribute_embeddings[patient_attributes_ids],
             "mod_embeddings": attribute_embeddings[mod_attributes_ids],
             "pos_embeddings": attribute_embeddings[pos_attributes_ids],
             })
    # break

  0%|          | 0/379 [00:00<?, ?it/s]

In [12]:
# ✅ Save it
detectives_features_dataset_path = os.path.join("data", "detectives_features_dataset.pt")
torch.save(detectives_features_dataset, detectives_features_dataset_path)

# ✅ Load it back
loaded = torch.load(detectives_features_dataset_path)
print(loaded[0])

{'file_name': '1811_Chateaubriand-François-Rene-de_Oeuvres-completes', 'char_name': 'le père babin', 'char_id': 3, 'label': 0, 'agent_lemmas': ['dire', 'savoir', 'voir', 'pouvoir', 'avouer', 'découvrir', 'voir', 'sentir', 'passer', 'voir', 'considérer', 'avoir', 'oublier', 'savoir', 'daigner', 'dire', 'rougir', 'trouver', 'achever', 'examiner', 'obtenir', 'pouvoir', 'avoir', 'pouvoir', 'avoir', 'attendre', 'avoir', 'arriver', 'allumer', 'embarquer', 'trouver', 'être', 'entreprendre', 'pouvoir', 'prendre', 'arriver', 'courir', 'congédier', 'voir', 'aller', 'sentir', 'suivre', 'songer', 'représenter', 'visiter', 'partir', 'charger', 'être', 'avoir', 'voir', 'perdre', 'mettre', 'laisser', 'occuper', 'figurer', 'promettre', 'ouvrir', 'bâtir', 'préparer', 'acheter', 'négliger', 'fondre', 'inviter', 'avoir', 'encourager', 'sortir', 'retrouver', 'savoir', 'transporter', 'apercevoir', 'prendre', 'amuser', 'attraper', 'venir', 'apercevoir', 'avoir', 'croire', 'sentir', 'tomber', 'rester', 'mour